# Buổi 5 - Pandas nâng cao I: Missing Data và kết hợp dữ liệu

Notebook này tổng hợp toàn bộ kiến thức của Buổi 5 theo slide môn học và bốn chương Pandas trong *Python Data Science Handbook*.

## Mục tiêu học tập

Sau khi chạy từ trên xuống dưới, bạn có thể:

1. Nhận biết và xử lý dữ liệu thiếu với `NaN`, `None`, `pd.NA`, `isna()`, `dropna()`, `fillna()`.
2. Tạo, truy cập và biến đổi `MultiIndex`; chuyển dữ liệu long ↔ wide với `stack()`/`unstack()`.
3. Ghép dữ liệu theo trục bằng `pd.concat()`, kiểm soát index trùng và cột không khớp.
4. Kết hợp dữ liệu theo khóa bằng `pd.merge()`/`.join()`, phân biệt one-to-one, many-to-one, many-to-many và bốn kiểu join.
5. Hoàn thiện hai case study: US Baby Names và US State Population.

> Cách chạy: chọn **Restart Kernel and Run All Cells**. Notebook tự tìm thư mục gốc repo nên chạy được cả khi Jupyter mở ở root hoặc trong `notebooks/`.


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

def find_repo_root(start: Path) -> Path:
    """Tìm root dựa trên thư mục dữ liệu của Buổi 5."""
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "datasets" / "buoi5").is_dir():
            return candidate
    raise FileNotFoundError("Không tìm thấy datasets/buoi5 từ thư mục hiện tại.")

ROOT = find_repo_root(Path.cwd())
DATA_DIR = ROOT / "datasets" / "buoi5"
SEED = 42
rng = np.random.default_rng(SEED)

{
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "repo_root": str(ROOT),
    "data_dir_exists": DATA_DIR.is_dir(),
}


{'python': '3.13.1',
 'numpy': '2.5.1',
 'pandas': '2.3.3',
 'repo_root': 'C:\\Hon\\Nam_3_HK1 2026_2027\\DataScience\\Hoan-Data-Science-Course',
 'data_dir_exists': True}

## 1. Xử lý dữ liệu thiếu (Missing Data)

Dữ liệu thực tế thường thiếu vì lỗi nhập liệu, nguồn không cung cấp hoặc hai bảng không khớp khóa.

- `None`: đối tượng Python; NumPy thường phải dùng `dtype=object`, chậm hơn và có thể làm phép tính lỗi.
- `np.nan`: sentinel số thực theo IEEE 754; khiến cột số nguyên truyền thống bị chuyển thành `float`.
- `pd.NA`: giá trị thiếu thống nhất cho các nullable dtype như `Int64`, `Float64`, `string`, `boolean`.

Hai nhóm quy ước phổ biến là **sentinel** (một giá trị đặc biệt biểu diễn thiếu) và **mask** (mảng Boolean đánh dấu vị trí thiếu). Nullable dtype của Pandas dùng hướng mask và nên được ưu tiên khi cần giữ đúng kiểu dữ liệu.


In [2]:
vals_none = np.array([1, None, 2, 3])
try:
    none_sum = vals_none.sum()
except TypeError as exc:
    none_sum = f"{type(exc).__name__}: {exc}"

series_default = pd.Series([1, None, 3], name="default")
series_nullable = pd.Series([1, None, 3], dtype="Int64", name="nullable")

print("NumPy có None:", vals_none, "| dtype:", vals_none.dtype)
print("Kết quả sum():", none_sum)
display(pd.concat([series_default, series_nullable], axis=1))
print("dtype mặc định:", series_default.dtype, "| nullable:", series_nullable.dtype)


NumPy có None: [1 None 2 3] | dtype: object
Kết quả sum(): TypeError: unsupported operand type(s) for +: 'int' and 'NoneType'


,default,nullable
0,1.00,1
1,NaN,<NA>
2,3.00,3


dtype mặc định: float64 | nullable: Int64


### 1.1 Phát hiện dữ liệu thiếu

Bước kiểm tra tối thiểu trước mọi biến đổi lớn:

- kích thước `shape`;
- kiểu dữ liệu `dtypes`;
- số và tỷ lệ missing theo cột;
- số dòng trùng `duplicated()`.

`isna()` và `isnull()` là hai tên tương đương; `notna()`/`notnull()` là phép phủ định.


In [3]:
students = pd.DataFrame({
    "id": pd.Series([1, 2, 3, 4, pd.NA], dtype="Int64"),
    "age": pd.Series([19, pd.NA, 21, 20, pd.NA], dtype="Int64"),
    "score": pd.Series([8.5, 9.0, pd.NA, 7.5, pd.NA], dtype="Float64"),
    "city": pd.Series(["HCM", "HN", "HCM", pd.NA, pd.NA], dtype="string"),
})

missing_report = pd.DataFrame({
    "count": students.isna().sum(),
    "percent": students.isna().mean().mul(100).round(1),
})

print("Shape:", students.shape)
display(students)
display(students.dtypes.rename("dtype").to_frame())
display(missing_report)
print("Tổng ô thiếu:", int(students.isna().sum().sum()))
print("Số dòng trùng hoàn toàn:", int(students.duplicated().sum()))


Shape: (5, 4)


,id,age,score,city
0,1,19,8.50,HCM
1,2,<NA>,9.00,HN
2,3,21,<NA>,HCM
3,4,20,7.50,<NA>
4,<NA>,<NA>,<NA>,<NA>


,dtype
id,Int64
age,Int64
score,Float64
city,string[python]


,count,percent
id,1,20.00
age,2,40.00
score,2,40.00
city,2,40.00


Tổng ô thiếu: 7
Số dòng trùng hoàn toàn: 0


### 1.2 Loại bỏ hay điền?

Không có một chiến lược đúng cho mọi cột.

- `dropna()`: phù hợp khi số dòng thiếu ít hoặc bản ghi không còn giá trị sử dụng.
- `thresh`: giữ dòng có đủ số trường hợp lệ, tránh xóa quá mạnh.
- `fillna()`: cần dựa trên ý nghĩa dữ liệu. Điểm thiếu không đồng nghĩa điểm 0.
- `ffill()`/`bfill()`: thường hợp với chuỗi thời gian, nhưng chỉ khi thứ tự quan sát có ý nghĩa.
- Nên giữ DataFrame gốc và tạo bản sao sau xử lý để dễ đối chiếu.


In [4]:
drop_examples = {
    "mặc định - bỏ dòng có bất kỳ NA": students.dropna(),
    "how='all' - chỉ bỏ dòng toàn NA": students.dropna(how="all"),
    "thresh=3 - giữ dòng có ít nhất 3 giá trị": students.dropna(thresh=3),
    "subset age, score": students.dropna(subset=["age", "score"]),
}

for title, frame in drop_examples.items():
    print(f"\n{title}: shape={frame.shape}")
    display(frame)



mặc định - bỏ dòng có bất kỳ NA: shape=(1, 4)


,id,age,score,city
0,1,19,8.50,HCM



how='all' - chỉ bỏ dòng toàn NA: shape=(4, 4)


,id,age,score,city
0,1,19,8.50,HCM
1,2,<NA>,9.00,HN
2,3,21,<NA>,HCM
3,4,20,7.50,<NA>



thresh=3 - giữ dòng có ít nhất 3 giá trị: shape=(4, 4)


,id,age,score,city
0,1,19,8.50,HCM
1,2,<NA>,9.00,HN
2,3,21,<NA>,HCM
3,4,20,7.50,<NA>



subset age, score: shape=(2, 4)


,id,age,score,city
0,1,19,8.50,HCM
3,4,20,7.50,<NA>


In [5]:
imputed = students.dropna(how="all").copy()
imputed["age"] = imputed["age"].fillna(int(imputed["age"].median()))
imputed["score"] = imputed["score"].fillna(imputed["score"].median())
imputed["city"] = imputed["city"].fillna("Unknown")

time_values = pd.Series([10, pd.NA, pd.NA, 13, pd.NA], dtype="Float64")
time_fill = pd.DataFrame({
    "gốc": time_values,
    "ffill": time_values.ffill(),
    "bfill": time_values.bfill(),
})

display(imputed)
display(time_fill)
print("Missing còn lại sau chiến lược điền:", int(imputed.isna().sum().sum()))


,id,age,score,city
0,1,19,8.50,HCM
1,2,20,9.00,HN
2,3,21,8.50,HCM
3,4,20,7.50,Unknown


,gốc,ffill,bfill
0,10.00,10.00,10.00
1,<NA>,10.00,13.00
2,<NA>,10.00,13.00
3,13.00,13.00,13.00
4,<NA>,13.00,<NA>


Missing còn lại sau chiến lược điền: 0


In [6]:
# Tự kiểm tra Module Missing Data
assert int(students.isna().sum().sum()) == 7
assert str(series_nullable.dtype) == "Int64"
assert imputed.isna().sum().sum() == 0
assert len(students.dropna(how="all")) == 4
print("✓ Các kiểm tra Missing Data đều đạt.")


✓ Các kiểm tra Missing Data đều đạt.


## 2. Hierarchical Indexing (MultiIndex)

`MultiIndex` cho phép một trục mang nhiều cấp nhãn. Ví dụ tự nhiên: **bang × năm** hoặc **người × chỉ số**.

Lợi ích:

- giữ dữ liệu nhiều chiều trong Series/DataFrame 2D;
- chọn theo một cấp bằng `.loc[]` hoặc `.xs()`;
- chuyển long ↔ wide bằng `unstack()` và `stack()`;
- giữ index có tên để code dễ đọc và ít nhầm cấp.


In [7]:
index = pd.MultiIndex.from_tuples(
    [
        ("California", 2010), ("California", 2020),
        ("New York", 2010), ("New York", 2020),
        ("Texas", 2010), ("Texas", 2020),
    ],
    names=["state", "year"],
)
populations = [37_253_956, 39_538_223, 19_378_102, 20_201_249, 25_145_561, 29_145_505]
pop_multi = pd.Series(populations, index=index, name="population").sort_index()

display(pop_multi.to_frame())
print("California qua các năm:")
display(pop_multi.loc["California"])
print("Tất cả bang trong năm 2010:")
display(pop_multi.xs(2010, level="year"))
print("Một giá trị cụ thể:", pop_multi.loc[("California", 2010)])


population
state      year            
California 2010    37253956
           2020    39538223
New York   2010    19378102
           2020    20201249
Texas      2010    25145561
           2020    29145505

California qua các năm:


year
2010    37253956
2020    39538223
Name: population, dtype: int64

Tất cả bang trong năm 2010:


state
California    37253956
New York      19378102
Texas         25145561
Name: population, dtype: int64

Một giá trị cụ thể: 37253956


In [8]:
population_long = pop_multi.rename("population").reset_index()
population_indexed = population_long.set_index(["state", "year"]).sort_index()
population_wide = population_indexed["population"].unstack("year")
population_round_trip = population_wide.stack().rename("population").sort_index()

print("Dạng long:")
display(population_long)
print("Dạng wide:")
display(population_wide)

pd.testing.assert_series_equal(
    population_round_trip,
    population_indexed["population"].sort_index(),
)
print("✓ stack(unstack(x)) khôi phục dữ liệu ban đầu khi không có ô thiếu.")


Dạng long:


,state,year,population
0,California,2010,37253956
1,California,2020,39538223
2,New York,2010,19378102
3,New York,2020,20201249
4,Texas,2010,25145561
5,Texas,2020,29145505


Dạng wide:


year,2010,2020
state,,
California,37253956,39538223
New York,19378102,20201249
Texas,25145561,29145505


✓ stack(unstack(x)) khôi phục dữ liệu ban đầu khi không có ô thiếu.


In [9]:
row_index = pd.MultiIndex.from_product(
    [[2025, 2026], [1, 2]], names=["year", "visit"]
)
column_index = pd.MultiIndex.from_product(
    [["An", "Binh"], ["HR", "Temp"]], names=["student", "measure"]
)
health_data = pd.DataFrame(
    [
        [72, 36.5, 80, 36.7],
        [74, 36.8, 79, 36.6],
        [71, 36.4, 82, 37.0],
        [73, 36.7, 81, 36.8],
    ],
    index=row_index,
    columns=column_index,
)

display(health_data)
print("Toàn bộ chỉ số của An:")
display(health_data["An"])
print("HR của An tại lần khám 2 năm 2026:", health_data.loc[(2026, 2), ("An", "HR")])


student     An       Binh      
measure     HR  Temp   HR  Temp
year visit                     
2025 1      72 36.50   80 36.70
     2      74 36.80   79 36.60
2026 1      71 36.40   82 37.00
     2      73 36.70   81 36.80

Toàn bộ chỉ số của An:


measure     HR  Temp
year visit          
2025 1      72 36.50
     2      74 36.80
2026 1      71 36.40
     2      73 36.70

HR của An tại lần khám 2 năm 2026: 73


## 3. Kết hợp theo trục với `pd.concat()`

`concat` **xếp chồng** các đối tượng theo hàng (`axis=0`) hoặc cột (`axis=1`). Nó không dò quan hệ khóa như SQL JOIN.

Các tham số quan trọng:

- `ignore_index=True`: đánh lại index từ 0 khi index cũ không còn ý nghĩa.
- `verify_integrity=True`: báo lỗi nếu index kết quả trùng.
- `keys=[...]`: thêm cấp nguồn gốc vào MultiIndex.
- `join="outer"|"inner"`: lấy hợp hoặc giao các nhãn trên trục còn lại.


In [10]:
df_a = pd.DataFrame({"name": ["An", "Binh"], "age": [25, 32]})
df_b = pd.DataFrame({"name": ["Chi", "Dung"], "age": [41, 29]})

concat_keep_index = pd.concat([df_a, df_b])
concat_reset_index = pd.concat([df_a, df_b], ignore_index=True)

try:
    pd.concat([df_a, df_b], verify_integrity=True)
except ValueError as exc:
    integrity_message = f"{type(exc).__name__}: phát hiện index trùng"

print("Giữ index gốc (có trùng):")
display(concat_keep_index)
print("Đánh lại index:")
display(concat_reset_index)
print(integrity_message)


Giữ index gốc (có trùng):


,name,age
0,An,25
1,Binh,32
0,Chi,41
1,Dung,29


Đánh lại index:


,name,age
0,An,25
1,Binh,32
2,Chi,41
3,Dung,29


ValueError: phát hiện index trùng


In [11]:
concat_with_source = pd.concat(
    [df_a, df_b],
    keys=["batch_1", "batch_2"],
    names=["source", "row"],
)

scores = pd.DataFrame({"score": [90, 85]}, index=df_a.index)
concat_columns = pd.concat([df_a, scores], axis="columns")

print("Dùng keys để giữ nguồn gốc:")
display(concat_with_source)
print("Nối theo cột - Pandas căn chỉnh bằng index:")
display(concat_columns)


Dùng keys để giữ nguồn gốc:


name  age
source  row           
batch_1 0      An   25
        1    Binh   32
batch_2 0     Chi   41
        1    Dung   29

Nối theo cột - Pandas căn chỉnh bằng index:


,name,age,score
0,An,25,90
1,Binh,32,85


In [12]:
df_left = pd.DataFrame(
    {"A": ["A1", "A2"], "B": ["B1", "B2"], "C": ["C1", "C2"]}
)
df_right = pd.DataFrame(
    {"B": ["B3", "B4"], "C": ["C3", "C4"], "D": ["D3", "D4"]}
)

print("join='outer' - hợp các cột, vị trí thiếu thành NA:")
display(pd.concat([df_left, df_right], ignore_index=True, join="outer"))
print("join='inner' - chỉ giữ giao các cột:")
display(pd.concat([df_left, df_right], ignore_index=True, join="inner"))


join='outer' - hợp các cột, vị trí thiếu thành NA:


,A,B,C,D
0,A1,B1,C1,NaN
1,A2,B2,C2,NaN
2,NaN,B3,C3,D3
3,NaN,B4,C4,D4


join='inner' - chỉ giữ giao các cột:


,B,C
0,B1,C1
1,B2,C2
2,B3,C3
3,B4,C4


### 3.1 Case study nhỏ: gộp US Baby Names nhiều năm

Bốn file mẫu chứa nguyên giá trị từ dữ liệu gốc, nhưng chỉ giữ một số tên nữ để notebook chạy nhanh. Mẫu này phù hợp để học quy trình; không dùng để suy rộng phân bố toàn nước Mỹ.

Quy trình: đọc từng file → thêm `year` trước khi ghép → `concat(ignore_index=True)` → tạo MultiIndex → `unstack()`.


In [13]:
BABY_DIR = DATA_DIR / "babynames_sample"
years = [1980, 1990, 2000, 2010]
name_frames = []

for year in years:
    one_year = pd.read_csv(
        BABY_DIR / f"yob{year}.txt",
        names=["name", "sex", "births"],
    )
    one_year["year"] = year
    name_frames.append(one_year)

names_all = pd.concat(name_frames, ignore_index=True)

baby_integrity = pd.Series({
    "rows": len(names_all),
    "columns": names_all.shape[1],
    "missing_cells": int(names_all.isna().sum().sum()),
    "duplicate_name_sex_year": int(
        names_all.duplicated(["name", "sex", "year"]).sum()
    ),
})
display(names_all)
display(baby_integrity.rename("value").to_frame())


,name,sex,births,year
0,Emma,F,533,1980
1,Jennifer,F,58375,1980
2,Isabella,F,36,1980
3,Emma,F,2410,1990
4,Jennifer,F,22218,1990
5,Isabella,F,215,1990
6,Emma,F,12533,2000
7,Jennifer,F,9386,2000
8,Isabella,F,6240,2000
9,Nevaeh,F,98,2000


,value
rows,14
columns,4
missing_cells,0
duplicate_name_sex_year,0


In [14]:
names_wide = (
    names_all
    .set_index(["name", "sex", "year"])["births"]
    .unstack("year")
    .sort_index()
)

print("Dạng wide - NaN nghĩa là tên không xuất hiện trong file mẫu của năm đó:")
display(names_wide)
print("Riêng tên Emma:")
display(names_wide.loc[("Emma", "F")].rename("births"))

print("Nếu bài toán quy ước 'không được ghi nhận' là 0:")
display(names_wide.fillna(0).astype("Int64"))


Dạng wide - NaN nghĩa là tên không xuất hiện trong file mẫu của năm đó:


,year,1980,1990,2000,2010
name,sex,,,,
Emma,F,533.00,"2,410.00","12,533.00","17,179.00"
Isabella,F,36.00,215.00,"6,240.00","22,731.00"
Jennifer,F,"58,375.00","22,218.00","9,386.00","2,601.00"
Nevaeh,F,NaN,NaN,98.00,"6,345.00"


Riêng tên Emma:


year
1980      533.00
1990    2,410.00
2000   12,533.00
2010   17,179.00
Name: births, dtype: float64

Nếu bài toán quy ước 'không được ghi nhận' là 0:


,year,1980,1990,2000,2010
name,sex,,,,
Emma,F,533,2410,12533,17179
Isabella,F,36,215,6240,22731
Jennifer,F,58375,22218,9386,2601
Nevaeh,F,0,0,98,6345


> **Có nên `fillna(0)` cho Baby Names?** Không nên làm tự động. Trong dữ liệu tên trẻ em, không xuất hiện có thể nghĩa là số lượt dưới ngưỡng công bố, không nhất thiết bằng 0. Ngược lại, với bảng đếm được xây dựng đầy đủ và quy ước rõ “không có quan sát = 0”, điền 0 có thể hợp lý. Ngữ nghĩa của cột quyết định chiến lược điền.


## 4. Kết hợp theo khóa với `pd.merge()` và `.join()`

| Công cụ | Cơ chế | Khi dùng |
|---|---|---|
| `pd.concat()` | Xếp chồng theo trục, căn chỉnh bằng index/columns | Nhiều file cùng cấu trúc |
| `pd.merge()` | Khớp giá trị khóa, tương tự SQL JOIN | Nhiều bảng có quan hệ |
| `.join()` | Cú pháp gọn để merge theo index | Khóa đã nằm ở index |

Ba quan hệ khóa:

- **one-to-one**: mỗi khóa xuất hiện tối đa một lần ở mỗi bảng;
- **many-to-one**: bảng trái có khóa lặp, bảng phải là bảng tra cứu khóa duy nhất;
- **many-to-many**: khóa lặp ở cả hai bảng, số dòng có thể tăng theo tích các kết hợp.

Dùng `validate=` để biến giả định quan hệ khóa thành kiểm tra có thể chạy.


In [15]:
employees = pd.DataFrame({
    "employee": ["Bob", "Jake", "Lisa", "Sue"],
    "group": ["Accounting", "Engineering", "Engineering", "HR"],
})
hire_dates = pd.DataFrame({
    "employee": ["Lisa", "Bob", "Jake", "Sue"],
    "hire_date": [2004, 2008, 2012, 2014],
})

one_to_one = pd.merge(
    employees,
    hire_dates,
    on="employee",
    validate="one_to_one",
)
display(one_to_one)


,employee,group,hire_date
0,Bob,Accounting,2008
1,Jake,Engineering,2012
2,Lisa,Engineering,2004
3,Sue,HR,2014


In [16]:
supervisors = pd.DataFrame({
    "group": ["Accounting", "Engineering", "HR"],
    "supervisor": ["Carly", "Guido", "Steve"],
})

many_to_one = pd.merge(
    one_to_one,
    supervisors,
    on="group",
    validate="many_to_one",
)
display(many_to_one)


,employee,group,hire_date,supervisor
0,Bob,Accounting,2008,Carly
1,Jake,Engineering,2012,Guido
2,Lisa,Engineering,2004,Guido
3,Sue,HR,2014,Steve


In [17]:
skills = pd.DataFrame({
    "group": ["Accounting", "Accounting", "Engineering", "Engineering", "HR", "HR"],
    "skill": ["math", "spreadsheets", "software", "math", "spreadsheets", "organization"],
})

many_to_many = pd.merge(
    employees,
    skills,
    on="group",
    validate="many_to_many",
)

print("Số dòng employees:", len(employees))
print("Số dòng sau many-to-many:", len(many_to_many))
display(many_to_many)


Số dòng employees: 4
Số dòng sau many-to-many: 8


,employee,group,skill
0,Bob,Accounting,math
1,Bob,Accounting,spreadsheets
2,Jake,Engineering,software
3,Jake,Engineering,math
4,Lisa,Engineering,software
5,Lisa,Engineering,math
6,Sue,HR,spreadsheets
7,Sue,HR,organization


### 4.1 Bốn kiểu join qua tham số `how`

- `inner` (mặc định): chỉ khóa có ở cả hai bảng.
- `outer`: hợp tất cả khóa, phần không khớp thành NA.
- `left`: giữ toàn bộ bảng trái.
- `right`: giữ toàn bộ bảng phải.

`inner` có thể âm thầm làm mất dòng. Luôn so sánh `shape`, kiểm tra khóa không khớp, và dùng `indicator=True` khi cần truy vết.


In [18]:
food = pd.DataFrame({
    "name": ["Peter", "Paul", "Mary"],
    "food": ["fish", "beans", "bread"],
})
drink = pd.DataFrame({
    "name": ["Mary", "Joseph"],
    "drink": ["wine", "beer"],
})

for how in ["inner", "outer", "left", "right"]:
    result = pd.merge(food, drink, on="name", how=how, indicator=True)
    print(f"how='{how}', shape={result.shape}")
    display(result)


how='inner', shape=(1, 4)

,name,food,drink,_merge
0,Mary,bread,wine,both


how='outer', shape=(4, 4)


,name,food,drink,_merge
0,Joseph,NaN,beer,right_only
1,Mary,bread,wine,both
2,Paul,beans,NaN,left_only
3,Peter,fish,NaN,left_only


how='left', shape=(3, 4)


,name,food,drink,_merge
0,Peter,fish,NaN,left_only
1,Paul,beans,NaN,left_only
2,Mary,bread,wine,both


how='right', shape=(2, 4)


,name,food,drink,_merge
0,Mary,bread,wine,both
1,Joseph,NaN,beer,right_only


In [19]:
population_small = pd.DataFrame({
    "state/region": ["CA", "CA", "NY"],
    "year": [2000, 2010, 2010],
    "population": [33_871_648, 37_253_956, 19_378_102],
})
abbrevs_small = pd.DataFrame({
    "state": ["California", "New York"],
    "abbreviation": ["CA", "NY"],
})

different_key_names = pd.merge(
    population_small,
    abbrevs_small,
    how="left",
    left_on="state/region",
    right_on="abbreviation",
    validate="many_to_one",
).drop(columns="abbreviation")

rank_left = pd.DataFrame({"name": ["Bob", "Jake"], "rank": [1, 2]})
rank_right = pd.DataFrame({"name": ["Bob", "Jake"], "rank": [3, 1]})
ranked = pd.merge(
    rank_left,
    rank_right,
    on="name",
    suffixes=("_left", "_right"),
    validate="one_to_one",
)

display(different_key_names)
display(ranked)


,state/region,year,population,state
0,CA,2000,33871648,California
1,CA,2010,37253956,California
2,NY,2010,19378102,New York


,name,rank_left,rank_right
0,Bob,1,3
1,Jake,2,1


In [20]:
left_indexed = pd.DataFrame({"x": [1, 2, 3]}, index=["a", "b", "c"])
right_indexed = pd.DataFrame({"y": [4, 5, 6]}, index=["a", "b", "d"])

joined_on_index = left_indexed.join(right_indexed, how="outer")
display(joined_on_index)


,x,y
a,1.00,4.00
b,2.00,5.00
c,3.00,NaN
d,NaN,6.00


## 5. Case study đầy đủ: US State Population

Ba bảng thật:

- `state-population.csv`: dân số theo mã vùng, nhóm tuổi và năm;
- `state-abbrevs.csv`: tên bang ↔ mã viết tắt;
- `state-areas.csv`: diện tích theo dặm vuông.

Mục tiêu: kiểm tra dữ liệu → merge không làm mất dòng → tìm khóa không khớp → sửa mã đặc biệt → ghép diện tích → tính mật độ dân số.


In [21]:
STATE_DIR = DATA_DIR / "state"
pop = pd.read_csv(STATE_DIR / "state-population.csv")
areas = pd.read_csv(STATE_DIR / "state-areas.csv")
abbrevs = pd.read_csv(STATE_DIR / "state-abbrevs.csv")

state_input_report = pd.DataFrame({
    "rows": [len(pop), len(areas), len(abbrevs)],
    "columns": [pop.shape[1], areas.shape[1], abbrevs.shape[1]],
    "missing_cells": [
        int(pop.isna().sum().sum()),
        int(areas.isna().sum().sum()),
        int(abbrevs.isna().sum().sum()),
    ],
    "duplicate_rows": [
        int(pop.duplicated().sum()),
        int(areas.duplicated().sum()),
        int(abbrevs.duplicated().sum()),
    ],
}, index=["population", "areas", "abbrevs"])

display(state_input_report)
display(pop.head())
display(areas.head())
display(abbrevs.head())


,rows,columns,missing_cells,duplicate_rows
population,2544,4,20,0
areas,52,2,0,0
abbrevs,51,2,0,0


,state/region,ages,year,population
0,AL,under18,2012,"1,117,489.00"
1,AL,total,2012,"4,817,528.00"
2,AL,under18,2010,"1,130,966.00"
3,AL,total,2010,"4,785,570.00"
4,AL,under18,2011,"1,125,763.00"


,state,area (sq. mi)
0,Alabama,52423
1,Alaska,656425
2,Arizona,114006
3,Arkansas,53182
4,California,163707


,state,abbreviation
0,Alabama,AL
1,Alaska,AK
2,Arizona,AZ
3,Arkansas,AR
4,California,CA


In [22]:
merged = pd.merge(
    pop,
    abbrevs,
    how="outer",
    left_on="state/region",
    right_on="abbreviation",
    validate="many_to_one",
    indicator=True,
).drop(columns="abbreviation")

unmatched_codes = (
    merged.loc[merged["state"].isna(), "state/region"]
    .dropna()
    .unique()
)

print("Shape population trước merge:", pop.shape)
print("Shape sau merge:", merged.shape)
print("Mã vùng chưa khớp:", unmatched_codes)
display(merged["_merge"].value_counts().rename("rows").to_frame())


Shape population trước merge: (2544, 4)
Shape sau merge: (2544, 6)
Mã vùng chưa khớp: ['PR' 'USA']


,rows
_merge,
both,2448
left_only,96
right_only,0


In [23]:
special_states = {
    "PR": "Puerto Rico",
    "USA": "United States",
}
merged["state"] = merged["state"].fillna(
    merged["state/region"].map(special_states)
)

final = pd.merge(
    merged,
    areas,
    on="state",
    how="left",
    validate="many_to_one",
)

missing_area = (
    final.loc[final["area (sq. mi)"].isna(), ["state/region", "state"]]
    .drop_duplicates()
    .sort_values("state/region")
)

print("Missing tên bang sau sửa:", int(final["state"].isna().sum()))
print("Vùng không có diện tích để tính density:")
display(missing_area)


Missing tên bang sau sửa: 0
Vùng không có diện tích để tính density:


,state/region,state
2160,USA,United States


In [24]:
analysis_base = final.dropna(
    subset=["population", "area (sq. mi)"]
).copy()

data2010 = (
    analysis_base
    .query("year == 2010 and ages == 'total'")
    .set_index("state")
)

density_total = (
    data2010["population"] / data2010["area (sq. mi)"]
).sort_values(ascending=False)
density_total.name = "people_per_sq_mile"

print("5 khu vực mật độ dân số cao nhất năm 2010:")
display(density_total.head().to_frame())
print("5 bang/khu vực mật độ thấp nhất năm 2010:")
display(density_total.tail().to_frame())


5 khu vực mật độ dân số cao nhất năm 2010:


,people_per_sq_mile
state,
District of Columbia,"8,898.90"
Puerto Rico,"1,058.67"
New Jersey,"1,009.25"
Rhode Island,681.34
Connecticut,645.60


5 bang/khu vực mật độ thấp nhất năm 2010:


,people_per_sq_mile
state,
South Dakota,10.58
North Dakota,9.54
Montana,6.74
Wyoming,5.77
Alaska,1.09


In [25]:
under18_2010 = (
    analysis_base
    .query("year == 2010 and ages == 'under18'")
    .set_index("state")
)
density_under18 = (
    under18_2010["population"] / under18_2010["area (sq. mi)"]
)
density_under18.name = "under18_per_sq_mile"

rank_comparison = pd.concat(
    [density_total, density_under18],
    axis="columns",
).dropna()
rank_comparison["rank_total"] = rank_comparison["people_per_sq_mile"].rank(
    ascending=False, method="min"
).astype(int)
rank_comparison["rank_under18"] = rank_comparison["under18_per_sq_mile"].rank(
    ascending=False, method="min"
).astype(int)
rank_comparison["rank_change"] = (
    rank_comparison["rank_under18"] - rank_comparison["rank_total"]
)

print("Các nơi thay đổi thứ hạng nhiều nhất khi chỉ xét dân số dưới 18 tuổi:")
display(
    rank_comparison
    .assign(abs_change=lambda d: d["rank_change"].abs())
    .sort_values("abs_change", ascending=False)
    .drop(columns="abs_change")
    .head(10)
)

california_wide = (
    analysis_base
    .query("state == 'California' and ages == 'total'")
    .set_index(["state", "year"])["population"]
    .unstack("year")
    .sort_index(axis="columns")
)
print("Dân số California theo năm - minh họa unstack():")
display(california_wide)


Các nơi thay đổi thứ hạng nhiều nhất khi chỉ xét dân số dưới 18 tuổi:


,people_per_sq_mile,under18_per_sq_mile,rank_total,rank_under18,rank_change
state,,,,,
Vermont,65.09,13.38,32,36,4
Texas,93.99,25.61,26,23,-3
Utah,32.68,10.28,43,40,-3
Oregon,39.00,8.79,40,42,2
Maine,37.51,7.72,41,43,2
Virginia,187.62,43.37,15,16,1
Louisiana,87.68,21.58,28,27,-1
Indiana,178.20,44.09,16,15,-1
Mississippi,61.32,15.57,33,32,-1


Dân số California theo năm - minh họa unstack():


year,1990,1991,1992,1993,1994,1995,1996,1997,1998,1999,...,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013
state,,,,,,,,,,,,,,,,,,,,,
California,"29,959,515.00","30,470,736.00","30,974,659.00","31,274,928.00","31,484,435.00","31,696,582.00","32,018,834.00","32,486,010.00","32,987,675.00","33,499,204.00",...,"35,574,576.00","35,827,943.00","36,021,202.00","36,250,311.00","36,604,337.00","36,961,229.00","37,333,601.00","37,668,681.00","37,999,878.00","38,332,521.00"


In [26]:
left_merge_demo = pd.merge(
    pop,
    abbrevs,
    how="left",
    left_on="state/region",
    right_on="abbreviation",
    indicator=True,
)
outer_merge_demo = pd.merge(
    pop,
    abbrevs,
    how="outer",
    left_on="state/region",
    right_on="abbreviation",
    indicator=True,
)

merge_mode_report = pd.DataFrame({
    "left": left_merge_demo["_merge"].value_counts().reindex(
        ["left_only", "right_only", "both"], fill_value=0
    ),
    "outer": outer_merge_demo["_merge"].value_counts().reindex(
        ["left_only", "right_only", "both"], fill_value=0
    ),
})
display(merge_mode_report)
print(
    "Trong bộ dữ liệu này, mọi mã ở bảng abbrevs đều có mặt trong population, "
    "nên outer không tạo right_only; khác biệt sẽ xuất hiện nếu bảng phải có khóa mới."
)


,left,outer
_merge,,
left_only,96,96
right_only,0,0
both,2448,2448


Trong bộ dữ liệu này, mọi mã ở bảng abbrevs đều có mặt trong population, nên outer không tạo right_only; khác biệt sẽ xuất hiện nếu bảng phải có khóa mới.


## 6. Checklist chống lỗi thường gặp

1. **Trước xử lý:** kiểm tra `shape`, `dtypes`, missing, duplicate và tính duy nhất của khóa.
2. **Missing data:** không điền 0 theo thói quen; ghi rõ ý nghĩa và giả định của chiến lược.
3. **MultiIndex:** đặt tên level, sắp xếp index khi cần slicing, kiểm tra round-trip long ↔ wide.
4. **Concat:** quyết định có giữ index nguồn không; dùng `verify_integrity` nếu index phải duy nhất.
5. **Merge:** khai báo `how`, `validate`, so sánh số dòng trước/sau và dùng `indicator` để tìm khóa không khớp.
6. **Tính toán:** chỉ loại missing ở các cột thật sự cần cho công thức, không `dropna()` toàn bảng một cách mù quáng.


In [27]:
# Kiểm tra tích hợp cuối notebook
assert DATA_DIR.is_dir()
assert names_all.isna().sum().sum() == 0
assert len(names_all) == sum(len(frame) for frame in name_frames)
assert names_wide.loc[("Emma", "F"), 2010] == 17_179
assert pd.isna(names_wide.loc[("Nevaeh", "F"), 1980])

assert len(merged) >= len(pop)
assert final["state"].isna().sum() == 0
assert density_total.index.is_unique
assert np.isfinite(density_total).all()
assert density_total.index[0] == "District of Columbia"
assert density_total.index[-1] == "Alaska"

print("✓ Toàn bộ kiểm tra tích hợp đều đạt.")


✓ Toàn bộ kiểm tra tích hợp đều đạt.


## 7. Bài tự luyện

1. Với `students`, thử `dropna(thresh=4)` và giải thích vì sao chỉ còn một số dòng.
2. Tạo MultiIndex `city × semester` cho điểm trung bình; dùng `.xs()`, `unstack()`, rồi `stack()`.
3. Thêm một file Baby Names mẫu mới, đảm bảo thêm cột `year` trước `concat`.
4. Cố tình tạo bảng tra cứu `abbrevs` có khóa trùng rồi chạy `validate="many_to_one"`; đọc thông báo lỗi.
5. Từ `rank_comparison`, giải thích ba nơi có `rank_change` lớn nhất bằng lời.
6. So sánh `how="inner"` với `how="outer"` trên hai bảng tự tạo có cả khóa chỉ-trái và chỉ-phải.


In [28]:
# Không gian làm bài tự luyện
# Ví dụ:
# students.dropna(thresh=4)


## 8. Ba kiến thức cần nhớ

1. **Missing không phải là 0.** Chọn `dropna` hay `fillna` theo ngữ nghĩa, và dùng nullable dtype để giữ đúng kiểu dữ liệu.
2. **`concat` và `merge` giải hai bài toán khác nhau.** `concat` xếp theo trục; `merge` khớp theo khóa và có thể làm tăng/mất dòng.
3. **Kiểm tra là một phần của pipeline.** `shape`, missing, duplicate, `validate`, `indicator` và các `assert` giúp kết quả có thể giải thích và chạy lại.

## Nguồn

- Slide môn học: [Buổi 5 - Pandas Nâng Cao I](../slides/buoi5_python_datascience.pdf)
- Jake VanderPlas, *Python Data Science Handbook*: [03.04 Missing Values](https://github.com/jakevdp/PythonDataScienceHandbook/blob/master/notebooks/03.04-Missing-Values.ipynb), [03.05 Hierarchical Indexing](https://github.com/jakevdp/PythonDataScienceHandbook/blob/master/notebooks/03.05-Hierarchical-Indexing.ipynb), [03.06 Concat and Append](https://github.com/jakevdp/PythonDataScienceHandbook/blob/master/notebooks/03.06-Concat-And-Append.ipynb), [03.07 Merge and Join](https://github.com/jakevdp/PythonDataScienceHandbook/blob/master/notebooks/03.07-Merge-and-Join.ipynb).
- Dữ liệu US States: [jakevdp/data-USstates](https://github.com/jakevdp/data-USstates).
- Dữ liệu US Baby Names: [davidrpugh/python-for-data-analysis](https://github.com/davidrpugh/python-for-data-analysis/tree/master/datasets/babynames).

Nội dung được diễn giải và tổ chức lại phục vụ mục tiêu học tập của Buổi 5; các ví dụ có thêm kiểm tra hiện đại như `validate`, `indicator` và nullable dtype.
